# Connectivity Graph Visualisation

Visualises the geographic **k-NN connectivity graph** used to constrain Ward hierarchical clustering in `cooccurrence_clustering_all.ipynb`.

**Prerequisite:** uncomment and run Cell 32 in `cooccurrence_clustering_all.ipynb` to write
`site_regions_cooccurrence_{N_CLUSTERS}.csv` before running this notebook.
That file provides site coordinates and cluster labels — the heavy drought matrix is *not* needed here.

In [ ]:
import os
import numpy as np
import pandas as pd
import re
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from sklearn.neighbors import kneighbors_graph
from scipy.sparse import coo_matrix
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import colorcet as cc
from scipy.spatial import ConvexHull
from shapely.geometry import MultiPoint
import shapely
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection

# ── Match these to cooccurrence_clustering_all.ipynb ──────────────────────────
CONUS_DIR   = os.path.dirname(os.path.abspath('connectivity_graph_viz.ipynb'))
K_NEIGHBORS = 11
N_CLUSTERS  = 10

SAVEFIG = False
FIG_DIR = CONUS_DIR   # directory for saved figures


# CONUS extent
CONUS_EXTENT = [-125, -66, 24, 50]  # [lon_min, lon_max, lat_min, lat_max]

def make_conus_ax(fig, pos=111, title=''):
    """Return a cartopy GeoAxes set to CONUS with standard features."""
    subplot_args = pos if isinstance(pos, tuple) else (pos,)
    ax = fig.add_subplot(*subplot_args, projection=ccrs.AlbersEqualArea(
        central_longitude=-96, central_latitude=37.5))
    ax.set_extent(CONUS_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,       facecolor='#f5f5f0', zorder=0)
    ax.add_feature(cfeature.OCEAN,      facecolor='#c8e0f0', zorder=0)
    ax.add_feature(cfeature.LAKES,      facecolor='#c8e0f0', zorder=1, alpha=0.6)
    ax.add_feature(cfeature.STATES,     linewidth=0.4,       zorder=2, edgecolor='gray')
    ax.add_feature(cfeature.COASTLINE,  linewidth=0.6,       zorder=3)
    ax.add_feature(cfeature.BORDERS,    linewidth=0.6,       zorder=3)
    if title:
        ax.set_title(title)
    return ax

## 1. Load cluster outputs

In [ ]:
csv_path = os.path.join(CONUS_DIR, f'site_regions_cooccurrence_ref_{N_CLUSTERS}.csv')
assert os.path.exists(csv_path), (
    f"File not found: {csv_path}\n"
    "Uncomment and run Cell 32 in cooccurrence_clustering_all.ipynb first."
)

site_meta = pd.read_csv(csv_path)
print(f"Loaded {len(site_meta):,} sites  |  {N_CLUSTERS} clusters")
print(site_meta.head())

## 2. Rebuild connectivity graph

The k-NN graph depends only on site coordinates — no drought data needed.

In [ ]:
coords = site_meta[['lat', 'lon']].values   # (n_sites, 2)
labels = site_meta['cluster'].values - 1    # 0-indexed for cmap

connectivity = kneighbors_graph(
    coords,
    n_neighbors=K_NEIGHBORS,
    include_self=False,
    mode='connectivity'
)
connectivity = 0.5 * (connectivity + connectivity.T)   # symmetrise

# Extract edges (upper triangle only to avoid duplicates)
coo = coo_matrix(connectivity)
mask = coo.row < coo.col
edge_rows = coo.row[mask]
edge_cols = coo.col[mask]

print(f"Sites: {len(coords):,}  |  Edges: {len(edge_rows):,}  (k={K_NEIGHBORS})")

## 3. Load edge phi coefficients

`phi_ij` is the **phi coefficient** (Pearson correlation of the binary drought indicators) between connected sites.  Run the save cell in `cooccurrence_clustering_all.ipynb` (inserted after Cell 11) to generate `edge_phi.npy`.

In [ ]:
phi_path = os.path.join(CONUS_DIR, 'edge_phi.npy')
assert os.path.exists(phi_path), (
    f"File not found: {phi_path}\n"
    "Run the 'Save edge phi' cell in cooccurrence_clustering_all.ipynb first."
)

phi_edges = np.load(phi_path)
print(f"Loaded {len(phi_edges):,} edge phi values")
print(f"Range: [{phi_edges.min():.3f}, {phi_edges.max():.3f}]  "
      f"median={np.median(phi_edges):.3f}")


## 4. Edges coloured by phi coefficient

Edge colour encodes the **phi coefficient** between the two connected sites — the Pearson correlation of their binary drought-indicator time series.  
Red = strong positive co-drought tendency; blue = anti-correlated.  
Node colour still shows cluster membership.

In [ ]:
from matplotlib.collections import LineCollection
import matplotlib.colors as mcolors

# phi_cmap  = plt.get_cmap('RdBu_r')
phi_cmap = plt.get_cmap('plasma')
phi_norm  = mcolors.TwoSlopeNorm(vmin=phi_edges.min(), vcenter=0, vmax=phi_edges.max())
# phi_norm  = mcolors.TwoSlopeNorm(vmin=-0.1, vcenter=0.45, vmax=1)
edge_rgba = phi_cmap(phi_norm(phi_edges))

fig = plt.figure(figsize=(16, 9))
ax  = make_conus_ax(fig, title=(
    f'k-NN graph — edges coloured by phi coefficient  (k={K_NEIGHBORS})\n'
    f'Yellow = high co-occurrence correlation  |  Blue = anti-correlated'
))

segments = [[(coords[r,1], coords[r,0]), (coords[c,1], coords[c,0])]
            for r, c in zip(edge_rows, edge_cols)]
lc = LineCollection(segments, linewidths=0.6, colors=edge_rgba, alpha=0.6,
                    transform=ccrs.PlateCarree(), zorder=3)
ax.add_collection(lc)

# Nodes — small, grey so edges stand out
ax.scatter(coords[:, 1], coords[:, 0], s=4, c='#333333',
           transform=ccrs.PlateCarree(), zorder=4)

# Colorbar
sm = plt.cm.ScalarMappable(cmap=phi_cmap, norm=phi_norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, orientation='vertical', shrink=0.5, pad=0.02,
             label='phi coefficient')

plt.tight_layout()
if SAVEFIG:
    plt.savefig(os.path.join(FIG_DIR, f'connectivity_graph_phi_k{K_NEIGHBORS}.png'),
                dpi=150, bbox_inches='tight')
plt.show()


## 5. Graph only — edges coloured by cluster

Each edge takes the colour of its source node's cluster.  Useful for seeing where cluster boundaries fall relative to the graph topology.

In [ ]:
from matplotlib.collections import LineCollection

cmap = cm.get_cmap('tab20', N_CLUSTERS)

fig = plt.figure(figsize=(16, 9))
ax  = make_conus_ax(fig, title=(
    f'k-NN connectivity graph  (k={K_NEIGHBORS}, {len(edge_rows):,} edges)\n'
    f'Nodes and edges coloured by cluster  |  N={N_CLUSTERS} clusters'
))

# Edges — one LineCollection for speed; colour = source-node cluster
segments    = [[(coords[r,1], coords[r,0]), (coords[c,1], coords[c,0])]
               for r, c in zip(edge_rows, edge_cols)]
edge_colors = [cmap(labels[r]) for r in edge_rows]
lc = LineCollection(segments, linewidths=0.4, colors=edge_colors, alpha=0.25,
                    transform=ccrs.PlateCarree(), zorder=3)
ax.add_collection(lc)

# Nodes on top
for cl in range(N_CLUSTERS):
    idx = np.where(labels == cl)[0]
    ax.scatter(
        coords[idx, 1], coords[idx, 0],
        s=10, color=cmap(cl), zorder=4,
        transform=ccrs.PlateCarree(),
        label=f'C{cl + 1} (n={len(idx)})'
    )

ax.legend(markerscale=2, fontsize=7, ncol=2, loc='lower left', framealpha=0.8)
plt.tight_layout()
if SAVEFIG:
    plt.savefig(os.path.join(FIG_DIR, f'connectivity_graph_k{K_NEIGHBORS}_c{N_CLUSTERS}.png'),
                dpi=150, bbox_inches='tight')
plt.show()


## 6. Graph only — edges in grey (no cluster colour)

Shows the raw graph structure without cluster colouring — useful for checking that the graph is well-connected and doesn't have isolated subgraphs.

In [ ]:
# from matplotlib.collections import LineCollection

# fig = plt.figure(figsize=(8, 5), dpi=150)
# ax  = make_conus_ax(fig, title=(
#     f'Geographic k-NN graph  (k={K_NEIGHBORS},  {len(edge_rows):,} edges,  {len(coords):,} sites)'
# ))

# segments = [[(coords[r,1], coords[r,0]), (coords[c,1], coords[c,0])]
#             for r, c in zip(edge_rows, edge_cols)]
# lc = LineCollection(segments, linewidths=0.3, colors='k', alpha=0.5,
#                     transform=ccrs.PlateCarree(), zorder=3)
# ax.add_collection(lc)

# ax.scatter(coords[:, 1], coords[:, 0], s=2, alpha=0.8, c='steelblue',
#            transform=ccrs.PlateCarree(), zorder=4)

# plt.tight_layout()
# if SAVEFIG:
#     plt.savefig(os.path.join(FIG_DIR, f'connectivity_graph_k{K_NEIGHBORS}_grey.png'),
#                 dpi=150, bbox_inches='tight')
# plt.show()


## 7. Zoomed regional view

Restrict to a bounding box to inspect graph structure in detail.
Edit `LON_RANGE` / `LAT_RANGE` to focus on any area of interest.

In [ ]:
# from matplotlib.collections import LineCollection

# # ── Bounding box — edit to zoom in ────────────────────────────────────────────
# LON_RANGE = (-125, -100)   # western CONUS / CRB
# LAT_RANGE = (  30,   50)

# # Sites inside the box
# in_box  = (
#     (coords[:, 1] >= LON_RANGE[0]) & (coords[:, 1] <= LON_RANGE[1]) &
#     (coords[:, 0] >= LAT_RANGE[0]) & (coords[:, 0] <= LAT_RANGE[1])
# )
# box_idx  = set(np.where(in_box)[0])
# box_mask = np.array([(r in box_idx and c in box_idx)
#                      for r, c in zip(edge_rows, edge_cols)])
# br, bc   = edge_rows[box_mask], edge_cols[box_mask]

# fig = plt.figure(figsize=(14, 8))
# ax  = make_conus_ax(fig, title=(
#     f'Zoomed graph  lon={LON_RANGE}  lat={LAT_RANGE}  '
#     f'(k={K_NEIGHBORS},  {len(br):,} edges shown)'
# ))
# ax.set_extent([*LON_RANGE, *LAT_RANGE], crs=ccrs.PlateCarree())

# segments = [[(coords[r,1], coords[r,0]), (coords[c,1], coords[c,0])]
#             for r, c in zip(br, bc)]
# lc = LineCollection(segments, linewidths=0.5, colors='k', alpha=0.25,
#                     transform=ccrs.PlateCarree(), zorder=3)
# ax.add_collection(lc)

# for cl in range(N_CLUSTERS):
#     idx = np.where(in_box & (labels == cl))[0]
#     if len(idx):
#         ax.scatter(
#             coords[idx, 1], coords[idx, 0],
#             s=18, color=cmap(cl), zorder=4,
#             transform=ccrs.PlateCarree(),
#             label=f'C{cl + 1} (n={len(idx)})'
#         )

# ax.legend(markerscale=1.5, fontsize=8, loc='lower left', framealpha=0.8)
# plt.tight_layout()
# if SAVEFIG:
#     plt.savefig(os.path.join(FIG_DIR, f'connectivity_graph_zoomed.png'),
#                 dpi=150, bbox_inches='tight')
# plt.show()


## 8. Cross-cluster edge count

How many k-NN edges *cross* a cluster boundary?  A high fraction means the graph allows many inter-cluster merges — a low fraction means clusters are well-separated in geographic graph space.

In [ ]:
cross = labels[edge_rows] != labels[edge_cols]
n_cross = cross.sum()
n_total = len(edge_rows)
print(f"Total edges : {n_total:,}")
print(f"Cross-cluster: {n_cross:,}  ({100 * n_cross / n_total:.1f}%)")
print(f"Within-cluster: {n_total - n_cross:,}  ({100 * (n_total - n_cross) / n_total:.1f}%)")

# Per-cluster breakdown
df_edges = pd.DataFrame({'src_cl': labels[edge_rows], 'dst_cl': labels[edge_cols]})
df_edges['cross'] = df_edges['src_cl'] != df_edges['dst_cl']
summary = df_edges.groupby('src_cl')['cross'].agg(['sum', 'count'])
summary.columns = ['cross_edges', 'total_edges']
summary['cross_pct'] = 100 * summary['cross_edges'] / summary['total_edges']
summary.index = [f'C{i+1}' for i in summary.index]
print()
print(summary.to_string())

## 10. Cluster outlines — geodesic convex hull

In [ ]:
cmap = cm.get_cmap('tab20', N_CLUSTERS)

fig = plt.figure(figsize=(14, 8))
ax  = make_conus_ax(fig, title=(
    f'Cluster extents — geodesic convex hull  '
    f'({N_CLUSTERS} clusters, k={K_NEIGHBORS})'
))

patches, colors = [], []

for cl in range(N_CLUSTERS):
    idx = np.where(labels == cl)[0]
    if len(idx) < 3:
        continue  # need ≥3 points for a hull

    pts = coords[idx]               # (n, 2) as [lat, lon]
    pts_lonlat = pts[:, [1, 0]]     # reorder to [lon, lat] for hull

    hull = ConvexHull(pts_lonlat)
    hull_verts = pts_lonlat[hull.vertices]          # ordered boundary vertices
    hull_closed = np.vstack([hull_verts, hull_verts[0]])  # close the polygon

    # Draw filled hull in map projection
    ax.fill(
        hull_closed[:, 0], hull_closed[:, 1],
        transform=ccrs.PlateCarree(),
        color=cmap(cl), alpha=0.20, zorder=2
    )
    ax.plot(
        hull_closed[:, 0], hull_closed[:, 1],
        transform=ccrs.PlateCarree(),
        color=cmap(cl), linewidth=1.2, zorder=3
    )

# Nodes on top
for cl in range(N_CLUSTERS):
    idx = np.where(labels == cl)[0]
    ax.scatter(
        coords[idx, 1], coords[idx, 0],
        s=8, color=cmap(cl), zorder=4,
        transform=ccrs.PlateCarree(),
        label=f'C{cl + 1} (n={len(idx)})'
    )

ax.legend(markerscale=1.5, fontsize=7, ncol=2, loc='lower left', framealpha=0.8)
plt.tight_layout()
if SAVEFIG:
    plt.savefig(os.path.join(FIG_DIR, f'cluster_hulls_k{K_NEIGHBORS}_c{N_CLUSTERS}.png'),
                dpi=150, bbox_inches='tight')
plt.show()


## 11. Cluster outlines — convex vs. concave hull

`HULL_MODE = 'convex'` replicates Section 10.  
`HULL_MODE = 'concave'` uses Shapely's `concave_hull` (Shapely ≥ 2.0).  
`CONCAVITY` controls tightness: `0.0` = maximally concave (tightest fit), `1.0` = convex hull.

In [ ]:
# ── Parameters ───────────────────────────────────────────────────────────────
HULL_MODE  = 'concave'   # 'convex' | 'concave'
CONCAVITY  = 0.4         # 0.0 = maximally concave (tightest), 1.0 = convex hull

cmap = cm.get_cmap('tab20', N_CLUSTERS)

fig = plt.figure(figsize=(14, 8))
ax  = make_conus_ax(fig, title=(
    f'Cluster extents — {HULL_MODE} hull'
    + (f'  (concavity={CONCAVITY})' if HULL_MODE == 'concave' else '')
    + f'\n{N_CLUSTERS} clusters, k={K_NEIGHBORS}'
))

def cluster_hull_coords(pts_lonlat, mode, ratio):
    """Return (lon_arr, lat_arr) of hull boundary, or None if too few points."""
    if len(pts_lonlat) < 3:
        return None
    if mode == 'concave':
        poly = shapely.concave_hull(MultiPoint(pts_lonlat), ratio=ratio)
        if poly.is_empty or poly.geom_type != 'Polygon':
            poly = MultiPoint(pts_lonlat).convex_hull  # fallback
        xy = np.array(poly.exterior.coords)
    else:
        hull = ConvexHull(pts_lonlat)
        verts = pts_lonlat[hull.vertices]
        xy = np.vstack([verts, verts[0]])
    return xy[:, 0], xy[:, 1]  # lons, lats

for cl in range(N_CLUSTERS):
    idx = np.where(labels == cl)[0]
    pts_lonlat = coords[idx][:, [1, 0]]  # [lon, lat]

    result = cluster_hull_coords(pts_lonlat, HULL_MODE, CONCAVITY)
    if result is None:
        continue
    lons, lats = result

    ax.fill(lons, lats, transform=ccrs.PlateCarree(),
            color=cmap(cl), alpha=0.20, zorder=2)
    ax.plot(lons, lats, transform=ccrs.PlateCarree(),
            color=cmap(cl), linewidth=1.2, zorder=3)

# Nodes
for cl in range(N_CLUSTERS):
    idx = np.where(labels == cl)[0]
    ax.scatter(
        coords[idx, 1], coords[idx, 0],
        s=8, color=cmap(cl), zorder=4,
        transform=ccrs.PlateCarree(),
        label=f'C{cl + 1} (n={len(idx)})'
    )

ax.legend(markerscale=1.5, fontsize=7, ncol=2, loc='lower left', framealpha=0.8)
plt.tight_layout()
if SAVEFIG:
    plt.savefig(
        os.path.join(FIG_DIR,
            f'cluster_hulls_{HULL_MODE}_k{K_NEIGHBORS}_c{N_CLUSTERS}.png'),
        dpi=150, bbox_inches='tight'
    )
plt.show()


## 9. Spatial heatmap — regional drought events per site

In [ ]:
# ── Load regional events ─────────────────────────────────────────────────────
events_path = os.path.join(CONUS_DIR, f'regional_events_obs_ref_{N_CLUSTERS}.csv')
assert os.path.exists(events_path), f"File not found: {events_path}"
obs_events = pd.read_csv(events_path)

# ── Count events per site from contrib_sites ─────────────────────────────────
def parse_sites(s):
    return [int(x) for x in re.findall(r'\d+', str(s))]

_exploded = obs_events.copy()
_exploded['_sites'] = _exploded['contrib_sites'].apply(parse_sites)
_exploded = _exploded.explode('_sites').rename(columns={'_sites': 'site_int'})
_exploded['site_int'] = _exploded['site_int'].astype(int)
site_event_counts = _exploded.groupby('site_int').size().reset_index(name='n_events')

# ── Merge with site coords from site_meta ────────────────────────────────────
plot_df = site_meta.copy()
plot_df['site_int'] = plot_df['site'].astype(str).str.lstrip('0').astype(int)
plot_df = plot_df.merge(site_event_counts, on='site_int', how='left').fillna(0)
plot_df['n_events'] = plot_df['n_events'].astype(int)

print(f"Sites: {len(plot_df):,}  |  Events: {len(obs_events):,}")
print(f"Event count range: {plot_df['n_events'].min()}–{plot_df['n_events'].max()}  "
      f"median={plot_df['n_events'].median():.0f}")

# ── Map ───────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 8))
ax  = make_conus_ax(fig, title=(
    f'Regional drought event frequency per site\n'
    f'({len(plot_df):,} sites  |  {len(obs_events):,} regional events  |  '
    f'{N_CLUSTERS} clusters)'
))

vmax = plot_df['n_events'].quantile(0.98)
sc = ax.scatter(
    plot_df['lon'], plot_df['lat'],
    c=plot_df['n_events'],
    cmap='YlOrRd',
    norm=mcolors.Normalize(vmin=0, vmax=vmax),
    s=18, alpha=0.85, linewidths=0.2, edgecolors='#555555',
    transform=ccrs.PlateCarree(),
    zorder=4
)


# Cluster borders
for cl in range(N_CLUSTERS):
    idx = np.where(labels == cl)[0]
    pts_lonlat = coords[idx][:, [1, 0]]  # [lon, lat]

    result = cluster_hull_coords(pts_lonlat, HULL_MODE, CONCAVITY)
    if result is None:
        continue
    lons, lats = result

    ax.plot(lons, lats, transform=ccrs.PlateCarree(),
            color=cmap(cl), linewidth=1.2, zorder=3)



cbar = fig.colorbar(sc, ax=ax, orientation='vertical', shrink=0.65, pad=0.02)
cbar.set_label('Number of regional drought events', fontsize=10)

plt.tight_layout()
if SAVEFIG:
    plt.savefig(os.path.join(FIG_DIR, f'site_event_heatmap_c{N_CLUSTERS}.png'),
                dpi=150, bbox_inches='tight')
plt.show()
